## Load Prepared Time Series

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/daily_patient_volume.csv")

df["appointmentday"] = pd.to_datetime(df["appointmentday"])
df.set_index("appointmentday", inplace=True)

df.head()

,patient_count,day_of_week
appointmentday,,
2016-04-29,2602.0,4
2016-04-30,0.0,5
2016-05-01,0.0,6
2016-05-02,3515.0,0
2016-05-03,3425.0,1


## Create Target Variable
We predict: *Tomorrow’s patient count*

In [3]:
df["target"] = df["patient_count"].shift(-1)

In [4]:
df.head()

,patient_count,day_of_week,target
appointmentday,,,
2016-04-29,2602.0,4,0.0
2016-04-30,0.0,5,0.0
2016-05-01,0.0,6,3515.0
2016-05-02,3515.0,0,3425.0
2016-05-03,3425.0,1,3195.0


## Create Lag Features (Most Important)

In [6]:
df["lag_1"] = df["patient_count"].shift(1)
df["lag_3"] = df["patient_count"].shift(3)
df["lag_7"] = df["patient_count"].shift(7)

## Create Rolling Features (Trend + Stability)

In [8]:
df["rolling_mean_3"] = df["patient_count"].rolling(window=3).mean()
df["rolling_std_3"] = df["patient_count"].rolling(window=3).std()

*Interpretation*
- Mean → trend
- Std → variability

## Create Calendar Features (Seasonality)

In [10]:
df["month"] = df.index.month
df["day_of_month"] = df.index.day
df["day_of_week"] = df.index.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

In [13]:
df.isnull().sum()

patient_count     0
day_of_week       0
target            1
lag_1             1
lag_3             3
lag_7             7
rolling_mean_3    2
rolling_std_3     2
month             0
day_of_month      0
is_weekend        0
dtype: int64

In [15]:
df = df.dropna()

In [17]:
df.describe()

,patient_count,day_of_week,target,lag_1,lag_3,lag_7,rolling_mean_3,rolling_std_3,month,day_of_month,is_weekend
count,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000
mean,2069.696970,3.030303,2088.515152,2065.636364,2150.515152,2019.363636,2060.686869,1217.292209,5.212121,15.424242,0.303030
std,1701.869936,2.083939,1716.803718,1698.260614,1656.759829,1663.709820,1140.086869,909.026998,0.415149,9.100241,0.466694
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,1.000000,0.000000
25%,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1095.000000,149.239182,5.000000,7.000000,0.000000
50%,3150.000000,3.000000,3198.000000,3150.000000,3195.000000,3084.000000,2183.333333,1818.653348,5.000000,15.000000,0.000000
75%,3508.000000,5.000000,3512.000000,3501.000000,3483.000000,3466.000000,3263.333333,1964.137555,5.000000,23.000000,1.000000
max,3819.000000,6.000000,3819.000000,3819.000000,3652.000000,3626.000000,3596.666667,2204.900678,6.000000,31.000000,1.000000


In [18]:
# Save the time series features
df.to_csv("../data/processed/time_series_features.csv")